# Medical PDF — Clinical Extraction v2 (Fabric only)
Reads the verified Hebrew PDF text from `silver.pdf_pages`. Deterministic section and date rules generate **candidates for review**, not clinical facts. The parser never calls a service, installs a dependency, or prints source text. Keep the notebook attached to `medical_documents` and run all cells in order.

Attach the target schema-enabled Lakehouse as default. Set the Lakehouse ID and input path in the first code cell. Run all cells in order. Start with the generated synthetic fixture.

- Silver: UTF-8 JSON in `Files/silver/clinical_entities/he_clinical_v2/` and a Delta index in `silver.clinical_candidates`. Exact source evidence stays in Silver.
- Gold: `visits`, `diagnoses`, `treatments`, `referrals`, `tests`, `medications`, and `date_mentions`. Filter these tables to `rule_version = 'he_clinical_v2'` for this run, and join to Silver by `candidate_id` for evidence. `event_date` is absent unless the date has an explicit clinical context; `date_mentions.event_date` is just a literal date, not a visit or treatment date. Tests and treatments may be mentions, not performed events. Visits are candidates only, and no other rows are linked to an inferred visit.
- Each row needs review. A zero count means the rules found no supported rows, not that the source contains none. Keep human review decisions separate from these generated tables. Gold's `extraction_runs` is written after read-back verification. Identical reruns upsert only this document/version in the generated tables. If rules or extracted source text change, increment the rule version to preserve the previous JSON.

In [ ]:
from pathlib import PurePosixPath
from uuid import UUID
import notebookutils

LAKEHOUSE_ID = '<target-lakehouse-guid>'
SOURCE_PATH = 'Files/raw/synthetic-medical-record.pdf'
EXPECTED_PAGES = 1  # Use the verified Silver page count for any other input.

try:
    expected_lakehouse = str(UUID(LAKEHOUSE_ID))
except (ValueError, TypeError) as error:
    raise ValueError('Set LAKEHOUSE_ID to the target default Lakehouse item GUID.') from error
if str(notebookutils.runtime.context['defaultLakehouseId']).lower() != expected_lakehouse:
    raise RuntimeError('Attach the configured Lakehouse as default and restart the session.')
path = PurePosixPath(SOURCE_PATH)
if path.parts[:2] != ('Files', 'raw') or '..' in path.parts or chr(92) in SOURCE_PATH or path.suffix.lower() != '.pdf':
    raise ValueError('SOURCE_PATH must be a relative PDF path under Files/raw/.')
SOURCE_PATH = path.as_posix()
if not isinstance(EXPECTED_PAGES, int) or isinstance(EXPECTED_PAGES, bool) or EXPECTED_PAGES < 1:
    raise ValueError('EXPECTED_PAGES must be the positive verified Silver page count.')


In [ ]:
from collections import Counter
from datetime import date, datetime, timezone
from hashlib import sha256
from itertools import chain
import json
import re
import unicodedata

VERSION = 'he_clinical_v2'
REVIEW = 'needs_review'
ENTITY = {
    'diagnoses': ('אבחנות', 'אבחנה', 'אבחנות רפואיות', 'אבחנות בביקור', 'אבחנות קבועות', 'אבחנות פעילות', 'diagnoses', 'diagnosis'),
    'medications': ('תרופות', 'תרופות קבועות', 'טיפול תרופתי', 'מרשמים', 'מרשם', 'medications', 'prescriptions'),
    'treatments': ('טיפולים', 'טיפול', 'מהלך הטיפול', 'טיפול שבוצע', 'טיפול מומלץ', 'פעולות', 'treatments', 'treatment', 'procedures'),
    'referrals': ('הפניות', 'הפניה', 'הפנייה', 'הפניות לבדיקות', 'הפניות למכונים', 'referrals', 'referral'),
    'tests': ('בדיקות', 'בדיקה', 'בדיקות מעבדה', 'בדיקות עזר', 'תוצאות מעבדה', 'תוצאות בדיקות', 'בדיקות דם', 'תוצאת בדיקה', 'שם הבדיקה', 'lab results', 'laboratory tests', 'tests', 'test result'),
    'visits': ('פרטי ביקור', 'תאריך ביקור', 'תאריך הביקור', 'מועד הביקור', 'מועד ביקור', 'זמן ביקור', 'ביקור', 'ביקורים', 'סיכום ביקור', 'תאריך מפגש', 'visit date', 'visit', 'encounter date'),
}
DATE_CONTEXT = {
    'visits': ('תאריך ביקור', 'תאריך הביקור', 'מועד הביקור', 'מועד ביקור', 'זמן ביקור', 'תאריך מפגש', 'visit date', 'encounter date'),
    'treatments': ('תאריך טיפול', 'תאריך הטיפול', 'מועד טיפול', 'מועד הטיפול', 'תאריך ביצוע', 'treatment date'),
    'referrals': ('תאריך הפניה', 'תאריך הפנייה', 'מועד ההפניה', 'referral date'),
    'tests': ('תאריך בדיקה', 'תאריך הבדיקה', 'מועד הבדיקה', 'test date'),
    'medications': ('תאריך מרשם', 'תאריך המרשם', 'prescription date'),
    'diagnoses': ('תאריך אבחנה', 'תאריך האבחנה', 'diagnosis date'),
}
SOURCE_LABELS = {label: group for group, labels in ENTITY.items() for label in labels}
DATE_LABELS = {label: group for group, labels in DATE_CONTEXT.items() for label in labels}
STOP = set(('תלונות', 'תלונה', 'תלונה עיקרית', 'תלונות וסיבת הפניה', 'אנמנזה', 'בדיקה גופנית', 'ממצאים', 'סיבת הביקור', 'סיכום', 'דיון', 'המלצות', 'דיון והמלצות', 'מעקב', 'תאריך', 'פרטי מטופל', 'תאריך לידה', 'תאריך הדפסה', 'עמוד', 'דף'))
DATE = re.compile(r'(?<![\d/.-])(?:((?:19|20)\d{2})-(\d{2})-(\d{2})|(\d{1,2})([/.-])(\d{1,2})\5((?:19|20)\d{2}))(?!\d|[/.-]\d)')
CLOCK = re.compile(r'^\s*((?:[01]?\d|2[0-3]):[0-5]\d)(?::[0-5]\d)?(?![\d:])')
CODE = re.compile(r'(?<![A-Z0-9.])(?:[A-Z][0-9]{2}(?:\.[A-Z0-9]{1,4})?|[0-9]{3}\.[0-9]{1,3})(?![A-Z0-9.])', re.I)
DOSE = re.compile(r'(?<!\d)(\d{1,5}(?:[.,]\d{1,3})?)\s*(mg|mcg|µg|μg|ml|g|iu|מ״ג|מק״ג|מ״ל|מ\"ג|מק\"ג|מ\"ל)(?![a-zא-ת])', re.I)
ASSERTION = (
    ('possible_negation', r'\b(?:ללא|נשלל(?:ה|ו)?|אין עדות|לא נמצא(?:ה|ו)?|ruled out|negative for|no evidence of)\b'),
    ('possible_or_suspected', r'\b(?:חשד ל|אפשרות ל|ייתכן|suspected|possible)\b'),
    ('historical_mention', r'\b(?:בעבר|היסטוריה של|history of|previously)\b'),
    ('plan_or_recommendation', r'\b(?:מומלץ|הומלץ|מתוכנן|טיפול מומלץ|planned|recommended)\b'),
)
IGNORE = re.compile(r'^(?:אין|לא|ללא|אין נתונים|לא ידוע|לא צוין|לא נרשם|לא קיימות|לא נמצאו|none|n/?a|not recorded)[.!]?$|^[-\s]+$', re.I)
VISIT_TAG = re.compile(r'(?:^|\s)(?:ביקור(?:ים)?|מפגש|visit|encounter)(?=\s|[:;,]|$)', re.I)
COMPACT_VISIT = re.compile(r'ביקור[\s־-]*(?:ראשון|נוסף)')
CLOCK_ANYWHERE = re.compile(r'(?<![\d:])((?:[01]?\d|2[0-3]):[0-5]\d)(?::[0-5]\d)?(?![\d:])')
TEST_MENTION = re.compile(r'בדיק(?:ות|ת|ה)')
TREATMENT_MENTION = re.compile(r'(?:ה?טיפול\s*[:：]\s*\S|(?:בוצע|ניתן|הומלץ)\s*טיפול)')
KEYS = (*ENTITY, 'date_mentions')

def clean(raw):
    return ' '.join(''.join(c for c in unicodedata.normalize('NFC', raw) if unicodedata.category(c) != 'Cf').split())

def split_label(text):
    key = text.strip(' :：').casefold()
    if key in SOURCE_LABELS or key in DATE_LABELS or key in STOP:
        return key, '', 'bare'
    left, colon, right = text.replace('：', ':').partition(':')
    key = left.strip().casefold()
    if colon and (key in SOURCE_LABELS or key in DATE_LABELS or key in STOP):
        return key, right.strip(), 'prefix'
    left, colon, right = text.replace('：', ':').rpartition(':')
    key = right.strip().casefold()
    if colon and (key in SOURCE_LABELS or key in DATE_LABELS or key in STOP):
        return key, left.strip(), 'suffix'
    return None, text, None

def dates_in(text):
    found = []
    for match in DATE.finditer(text):
        if match[1]:
            y, m, d = match[1], match[2], match[3]
        else:
            y, m, d = match[7], match[6], match[4]
        try:
            dt = date(int(y), int(m), int(d))
        except ValueError:
            continue
        tm = CLOCK.match(text[match.end():])
        found.append((match.group(0), dt.isoformat(), tm[1].zfill(5) if tm else None))
    return found

def evidence_id(*parts):
    return sha256('|'.join(map(str, parts)).encode('utf-8')).hexdigest()

def parse_page(doc, page, text):
    section = None
    candidates = []
    for line_no, raw in enumerate(text.splitlines(), 1):
        line = clean(raw)
        if not line or len(line) > 600:
            section = None
            continue
        label, content, layout = split_label(line)
        dates = dates_in(line)
        if label in STOP:
            section = None
        date_kind = DATE_LABELS.get(label)
        kind = SOURCE_LABELS.get(label)
        if date_kind and not kind:
            section = None
        heading_no = line_no if label else None
        rule = f'label_{layout}' if label else None
        if kind:
            section = (kind, line_no, label) if not content else None
        elif not label and section:
            active, anchor, heading = section
            # Do not extend across blank lines, unlabeled dates, or what looks like another heading.
            if line_no - anchor > 8 or dates or re.match(r'^.{1,36}:\s*$', line):
                section = None
            elif active != 'visits' and (line_no == anchor + 1 or (active in ('diagnoses', 'medications') and re.search(r'[a-zA-Z]', line))):
                kind, heading_no, label, rule = active, anchor, heading, 'heading_row_candidate'
        compact_visit_date = None
        if not kind and not date_kind and label not in STOP:
            clocks = CLOCK_ANYWHERE.findall(line) if len(dates) == 1 and COMPACT_VISIT.search(line) else []
            if len(clocks) == 1:
                kind, content, rule = 'visits', line, 'compact_visit_timestamp'
                compact_visit_date = (dates[0][0], dates[0][1], clocks[0].zfill(5))
            elif dates and VISIT_TAG.search(line):
                kind, content, rule = 'visits', line, 'date_with_visit_word'
            elif TEST_MENTION.search(line):
                kind, content, rule = 'tests', line, 'test_term_mention'
            elif TREATMENT_MENTION.search(line):
                kind, content, rule = 'treatments', line, 'treatment_label_mention'
        if date_kind and len(dates) == 1:
            kind = date_kind
            rule = 'explicit_date_label'
        label_hint = next((hint for hint, pattern in ASSERTION if re.search(pattern, line, re.I)), 'undetermined')

        def emit(category, ordinal, description, method, date_value=None, item_code=None, dose=None):
            literal, event_date, event_time = date_value or (None, None, None)
            candidates.append(dict(
                candidate_id=evidence_id(doc, VERSION, category, page, line_no, ordinal),
                document_id=doc, rule_version=VERSION, category=category,
                page_number=page, line_number=line_no, heading_line_number=heading_no,
                section_label=label, description=description, event_date=event_date, event_time=event_time,
                date_literal=literal, code_candidate=item_code,
                dose_value_text=dose[1] if dose else None, dose_unit=dose[2] if dose else None,
                assertion_hint=label_hint, extraction_rule=method, review_status=REVIEW,
                evidence_text=raw, evidence_sha256=sha256(raw.encode('utf-8')).hexdigest(),
            ))

        if kind and content and not IGNORE.fullmatch(content) and (kind != 'visits' or dates):
            codes = list(CODE.finditer(content)) if kind == 'diagnoses' else []
            doses = list(DOSE.finditer(content)) if kind == 'medications' else []
            dated = compact_visit_date or (dates[0] if len(dates) == 1 and (date_kind == kind or kind == 'visits') else None)
            emit(kind, 0, content, rule, dated, codes[0].group(0).upper() if len(codes) == 1 else None, doses[0] if len(doses) == 1 else None)
        for ordinal, value in enumerate(dates):
            emit('date_mentions', ordinal, None, 'literal_date', value)
    return candidates

fixture = 'אבחנות\nSYNTHETIC A00.1\n\nתרופות: PLACEHOLDER 25 mg\nתאריך הטיפול: 29/02/2024\n31/02/2024\n01/03/2024 08:30\n02/03/2024 :תאריך ביקור\nבדיקות: ללא ממצא\nהפניות: הומלץ לדוגמה'
sample = parse_page('synthetic', 1, fixture)
tallies = Counter(r['category'] for r in sample)
assert tallies == {'diagnoses': 1, 'medications': 1, 'treatments': 1, 'visits': 1, 'tests': 1, 'referrals': 1, 'date_mentions': 3}
assert next(r for r in sample if r['category'] == 'diagnoses')['code_candidate'] == 'A00.1'
assert next(r for r in sample if r['category'] == 'medications')['dose_value_text'] == '25'
assert next(r for r in sample if r['category'] == 'treatments')['event_date'] == '2024-02-29'
assert next(r for r in sample if r['category'] == 'visits')['event_date'] == '2024-03-02'
assert next(r for r in sample if r['category'] == 'tests')['assertion_hint'] == 'possible_negation'
assert next(r for r in sample if r['event_date'] == '2024-03-01')['event_time'] == '08:30'
assert parse_page('synthetic', 1, 'תרופות\nאין\n\nUNRELATED') == []
assert parse_page('synthetic', 1, 'טיפול: דוגמה עם 05/07/2024 ו-06/07/2024')[0]['event_date'] is None
assert [d[1] for d in dates_in('07.06.2024. 2024-06-09; 31/02/2024')] == ['2024-06-07', '2024-06-09']
assert parse_page('synthetic', 1, 'אבחנות: A00.1 A00.2')[0]['code_candidate'] is None
compact = parse_page('synthetic', 1, 'ביקורנוסף 15/05/2024 לדוגמה 11:45')
assert compact[0]['extraction_rule'] == 'compact_visit_timestamp' and compact[0]['event_time'] == '11:45'
assert parse_page('synthetic', 1, 'בדיקתדוגמה')[0]['extraction_rule'] == 'test_term_mention'
assert parse_page('synthetic', 1, 'הומלץ טיפול לדוגמה')[0]['assertion_hint'] == 'plan_or_recommendation'
assert parse_page('synthetic', 1, fixture) == sample
print('Synthetic parser checks passed.')

In [ ]:
from pyspark.sql import functions as F

bronze = (spark.table('bronze.pdf_documents').where(F.col('source_path') == SOURCE_PATH)
          .select('document_id', 'source_path').take(2))
if len(bronze) != 1 or not re.fullmatch(r'[0-9a-f]{64}', bronze[0].document_id or ''):
    raise RuntimeError('Expected one valid Bronze manifest row for this input. Run the ingestion notebook first.')
DOCUMENT_ID = bronze[0].document_id
page_rows = (spark.table('silver.pdf_pages').where(F.col('document_id') == DOCUMENT_ID)
             .select('page_number', 'extraction_status', 'text').orderBy('page_number').take(EXPECTED_PAGES + 1))
if [r.page_number for r in page_rows] != list(range(1, EXPECTED_PAGES + 1)):
    raise RuntimeError('Silver page sequence is incomplete or contains duplicates.')
if any(r.extraction_status != 'text_extracted' or not r.text or len(r.text) > 100_000 for r in page_rows):
    raise RuntimeError('Silver page text does not pass the extraction preconditions.')
raw_lines = {r.page_number: r.text.splitlines() for r in page_rows}
rows = list(chain.from_iterable(parse_page(DOCUMENT_ID, r.page_number, r.text) for r in page_rows))
ids = {r['candidate_id'] for r in rows}
counts = {k: 0 for k in KEYS} | dict(Counter(r['category'] for r in rows))
if not rows or len(ids) != len(rows) or set(counts) != set(KEYS):
    raise RuntimeError('No candidates found, or candidate keys/categories failed validation.')
for row in rows:
    page_lines = raw_lines[row['page_number']]
    if (row['review_status'] != REVIEW or page_lines[row['line_number'] - 1] != row['evidence_text'] or
        (row['heading_line_number'] is not None and not 0 < row['heading_line_number'] <= row['line_number'])):
        raise RuntimeError('Candidate source evidence failed read-back validation.')

json_dir = f'Files/silver/clinical_entities/{VERSION}'
json_path = f'{json_dir}/{DOCUMENT_ID}.json'
source_text_sha256 = sha256(json.dumps([r.text for r in page_rows], ensure_ascii=False).encode('utf-8')).hexdigest()
payload = dict(schema_version=1, rule_version=VERSION, document_id=DOCUMENT_ID,
               source_pdf_path=bronze[0].source_path, source_table='silver.pdf_pages', source_pages=EXPECTED_PAGES,
               source_text_sha256=source_text_sha256, review_status=REVIEW, counts=counts, candidates=rows)
serialized = (json.dumps(payload, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
if not notebookutils.fs.mkdirs(json_dir):
    raise OSError('Could not prepare the Silver clinical JSON folder.')
exists = notebookutils.fs.exists(json_path)
if not exists and not notebookutils.fs.put(json_path, serialized.decode('utf-8'), False):
    raise OSError('Could not write Silver clinical JSON.')
# Compare the full file bytes, not a text preview or a cached mount lookup.
stored = spark.read.format('binaryFile').load(json_path).select('length', F.sha2('content', 256).alias('digest')).take(2)
if len(stored) != 1 or stored[0].length != len(serialized) or stored[0].digest != sha256(serialized).hexdigest():
    if exists:
        raise RuntimeError('Existing clinical JSON differs. Increase the rule version instead of overwriting it.')
    raise RuntimeError('Clinical JSON did not match on read-back.')
print({'source_pages': EXPECTED_PAGES, 'candidates_by_category': counts, 'silver_json_bytes': len(serialized)})

In [ ]:
from delta.tables import DeltaTable

BASE_DDL = ('candidate_id STRING, document_id STRING, rule_version STRING, category STRING, '
            'page_number INT, line_number INT, heading_line_number INT, section_label STRING, description STRING, '
            'event_date DATE, event_time STRING, date_literal STRING, code_candidate STRING, '
            'dose_value_text STRING, dose_unit STRING, assertion_hint STRING, extraction_rule STRING, review_status STRING, '
            'evidence_sha256 STRING, evidence_text STRING')
base = spark.createDataFrame([
    r | {'event_date': date.fromisoformat(r['event_date']) if r['event_date'] else None} for r in rows
], BASE_DDL)
predicate = f"document_id = '{DOCUMENT_ID}' AND rule_version = '{VERSION}'"

def persist_slice(frame, table):
    if spark.catalog.tableExists(table):
        if spark.table(table).schema.names != frame.schema.names or any(
            a.dataType != b.dataType for a, b in zip(spark.table(table).schema, frame.schema)
        ):
            raise RuntimeError(f'Schema mismatch: {table}. No overwrite was performed on this table.')
        keys = ('document_id', 'rule_version', 'candidate_id') if 'candidate_id' in frame.columns else ('document_id', 'rule_version')
        match = ' AND '.join(f'target.{key} = incoming.{key}' for key in keys)
        (DeltaTable.forName(spark, table).alias('target').merge(frame.alias('incoming'), match)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        frame.coalesce(1).write.format('delta').mode('errorifexists').saveAsTable(table)

persist_slice(base, 'silver.clinical_candidates')
spark.sql('CREATE SCHEMA IF NOT EXISTS gold')
for kind in KEYS:
    typed = base.where(F.col('category') == kind).drop('category', 'evidence_text')
    if kind != 'diagnoses':
        typed = typed.drop('code_candidate')
    if kind != 'medications':
        typed = typed.drop('dose_value_text', 'dose_unit')
    persist_slice(typed, f'gold.{kind}')
print({'gold_candidate_tables_written': len(KEYS)})

In [ ]:
def candidate_keys(df):
    return df.select('candidate_id', 'category', 'page_number', 'line_number', 'heading_line_number',
                     'review_status', 'event_date', 'evidence_sha256')

def validate_keys(df, expected, label):
    expected_keys = {tuple(r[field] for field in ('candidate_id', 'category', 'page_number', 'line_number',
                      'heading_line_number', 'review_status', 'event_date', 'evidence_sha256')) for r in expected}
    stored = [tuple(r) for r in candidate_keys(df).take(len(expected) + 1)]
    if len(stored) != len(expected) or set(stored) != expected_keys:
        raise RuntimeError(f'Stored keys, source references, dates, or review status differ: {label}.')

typed_rows = [r | {'event_date': date.fromisoformat(r['event_date']) if r['event_date'] else None} for r in rows]
validate_keys(spark.table('silver.clinical_candidates').where(predicate), typed_rows, 'silver.clinical_candidates')
gold_keys = [candidate_keys(spark.table(f'gold.{kind}').where(predicate).withColumn('category', F.lit(kind))) for kind in KEYS]
combined = gold_keys[0]
for frame in gold_keys[1:]:
    combined = combined.unionByName(frame)
validate_keys(combined, typed_rows, 'Gold candidate tables')

completion_ddl = ('document_id STRING, rule_version STRING, source_pages INT, candidate_count BIGINT, '
                  'silver_json_path STRING, silver_json_sha256 STRING, counts_json STRING, checked_at TIMESTAMP, review_status STRING')
marker = spark.createDataFrame([(DOCUMENT_ID, VERSION, EXPECTED_PAGES, len(rows), json_path,
                                sha256(serialized).hexdigest(), json.dumps(counts, sort_keys=True),
                                datetime.now(timezone.utc), REVIEW)], completion_ddl)
persist_slice(marker, 'gold.extraction_runs')
saved = spark.table('gold.extraction_runs').where(predicate).select('candidate_count', 'silver_json_sha256').take(2)
if len(saved) != 1 or saved[0].candidate_count != len(rows) or saved[0].silver_json_sha256 != sha256(serialized).hexdigest():
    raise RuntimeError('Gold extraction run could not be verified.')
print({'verified': True, 'source_pages': EXPECTED_PAGES, 'candidates': len(rows), 'counts': counts, 'review_status': REVIEW})